# Routing the Locomizer commutes of London: car, bicycle, e-bike and transit

The London city notebook (`uk/commuting_london.ipynb`) gives every panel user a home cell and a work cell (H3 level 8) from two months of
signals. This notebook turns those home-to-work pairs into travel times by mode, the way the US metros are routed in the `cafein` project
(`us-commuting/cities/<city>/05.3*-lodes-routing-<city>.ipynb`) and the way London was routed once before on the WorldCommuting-OD grid
(`cafein/exploratory/04.2-london-routing-first-steps.ipynb`, 2026-11-02). What is reused from there, and what is new:

| | Reused | New here |
|---|---|---|
| Flows | – | Locomizer home -> work pairs per level-8 hexagon (step 1), instead of LODES tracts or the WorldCommuting grid |
| Routing points | the rule: one point per unit, placed where people live, snapped to a street inside the unit, at most 1 km away | the unit is the hexagon; the home point is the population-weighted mean of the census Output Area centroids inside it |
| Streets | cafein `StreetNetwork.from_osm` | the Geofabrik **England** extract clipped to the FUA, since Greater London alone does not cover the commuter belt |
| Transit | the two sources of 04.2: Bus Open Data Service through transitio + the gb-transit rail and TfL file | the bus feed is cropped to the FUA polygon, not to transitio's London polygon |
| Modes | car free-flow, car with rush-hour intersection delays, bicycle, e-bike, transit, as in Seattle | – |

**Why London has no single GTFS feed.** Transport for London publishes its timetables as TransXChange and through its own API, not as
GTFS, and National Rail publishes CIF files. The two usable GTFS sources are (a) the Department for Transport's Bus Open Data Service, a
national bus feed that also carries TfL buses, Underground, DLR and trams since 2023, and (b) the gb-transit project of Planar Network,
which converts the National Rail timetable and TfL's journey-planner timetables to GTFS every night. 04.2 combined the two; 04.2.1
compared them with the TravelWhiz feeds and found the transit times within 2 minutes on average, so the 04.2 pair is kept.

| Step | What we do | Job |
|---|---|---|
| 1. Flows | home and work cell per user with the city-notebook rule, summed to hexagon pairs; saved once | full node (the two UK user-day tables are 38 and 44 GB) |
| 2. Routing points | one point per home hexagon (where its residents live) and per work hexagon (its centre) | routing job |
| 3. Street network | England extract clipped to the FUA, built once and saved on scratch | routing job |
| 4. Snapping | each point moved to the nearest street inside its own hexagon, at most 1 km | routing job |
| 5. Street modes | car (free-flow and rush), bicycle, e-bike between all routing points, joined to the flows | routing job |
| 6. Timetables | BODS buses cropped to the FUA through transitio, plus the gb-transit rail and TfL file; what is inside | routing job |
| 7. Transit | the cafein transit network and the transit times at 08:30 on a Tuesday, joined next to the street modes | routing job |
| 8. Summary | share of commuters reached and minutes by mode, median per home hexagon, maps | routing job |

Step 1 runs when its output file is missing and is skipped afterwards, so the notebook can be executed twice: once on a full node
(`routing/jobs/london-flows.sbatch`) and then on a normal node (`routing/jobs/london-routing.sbatch`).

In [ ]:
import json, os, time, urllib.request, zipfile
from pathlib import Path

import duckdb, h3
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.ops import nearest_points

import transitio
from cafein import StreetNetwork, TransportNetwork, TravelTimeMatrix
from cafein._streaming import read_shards

## Parameters

Everything city-specific sits here. `AREA` chooses between the Urban Audit 2020 functional urban area (`"fua"`, 6,978 km², the metro
commuters of the city notebook) and Greater London (`"city"`, 1,576 km²). The FUA is the default because the US metros are routed at
metro scale and 60 % of the panel's London commuters live or work outside the 33 boroughs.

In [ ]:
CITY_KEY, CITY = "london", "London"
COUNTRY_CODE = "GBR"
PERIOD = "2024-05_2024-06"; MONTHS = PERIOD.split("_")
H3_RES = 8                                   # resolution of the user-day tables and of the home/work rule (city notebooks)
H_MIN, W_MIN = 10, 4                         # home_days and work_days thresholds of the city notebooks
HOLIDAYS = {"2024-05": [6, 27], "2024-06": []}   # UK bank holidays
NIGHT_HOURS = [22, 23, 0, 1, 2, 3, 4, 5, 6]; WORK_HOURS = [9, 10, 11, 12, 13, 14, 15, 16]; WORKDAYS = [1, 2, 3, 4, 5]
AREA = "fua"                                 # "fua" (Urban Audit 2020 FUA UK001L3) or "city" (Greater London, the 33 boroughs dissolved)
FUA_CODE = "UK001L3"
MIN_USERS_HOME = 1                           # home hexagons with fewer commuters than this are left out of the routing (1 = keep all)
STRONGEST = 20                               # pairs shown in the tables and routed in the smoke test

# ---- Locomizer side (project_2022768) ----
LOCO = Path(f"/scratch/project_2022768/{os.environ['USER']}/locomizer")
UD_PATHS = {m: LOCO / f"{COUNTRY_CODE}_userday_h3_{H3_RES}_{m}" / "userday.parquet" for m in MONTHS}
FUA_GEOJSON = LOCO / "boundaries" / "URAU_RG_100K_2020_4326_FUA.geojson"
CITY_GEOJSON = LOCO / "boundaries" / "URAU_RG_100K_2020_4326_CITIES.geojson"     # boroughs UK101C1..UK133C1, dissolved below
OA_CENTROIDS = LOCO / "boundaries" / "uk_oa" / "uk_oa_population_centroids.parquet"   # one row per census OA: lon, lat, population
RESULTS = LOCO / "results" / COUNTRY_CODE / f"{CITY_KEY}_{PERIOD}" / "routing"; RESULTS.mkdir(parents=True, exist_ok=True)
OD_FILE = RESULTS / f"od_h3_{H3_RES}.parquet"                                   # step 1 output: home_cell, work_cell, users

# ---- routing side (project_2010423, shared with the cafein project) ----
DATA = Path("/scratch/project_2010423/data")
OSM_URL = "https://download.geofabrik.de/europe/united-kingdom/england-latest.osm.pbf"
OSM_PBF = DATA / "osm" / "england-latest.osm.pbf"
COUNTRY = "GB"                               # legal default speed limits for ways without a maxspeed tag
METRIC_CRS = "EPSG:27700"                    # British National Grid, metres
SNAP_DISTANCE_M = 1000
WORK = Path("/scratch/project_2010423/carbon-advantage/commuting") / f"london-locomizer-{AREA}"; WORK.mkdir(parents=True, exist_ok=True)
STREET_ARTIFACT = WORK / "streets.cafein"
TRANSIT_ARTIFACT = WORK / "transit.cafein"
MATRICES = WORK / "matrices"; MATRICES.mkdir(exist_ok=True)

# Street modes as in the US metros: car free-flow and with cafein's rush-hour intersection delays (Jaakkola 2013),
# bicycle at cafein's 15.3 km/h, e-bike at 17.4 km/h (Schleinitz et al. 2017); no elevation model, so no slopes.
CAR_PROFILE = "rush"
STREET_QUERIES = {
    "car_freeflow": {"folder": "car", "query": {"transport_mode": "car"}},
    "car": {"folder": f"car-{CAR_PROFILE}", "query": {"transport_mode": "car", "intersection_delays": True, "profile": CAR_PROFILE}},
    "bicycle": {"folder": "bicycle", "query": {"transport_mode": "bicycle"}},
    "e_bike": {"folder": "e_bike", "query": {"transport_mode": "e_bike", "speeds": {"e_bike": 17.4}}},
}
STREET_MODES = list(STREET_QUERIES)

# Transit: the same departure as every other city (a normal Tuesday, 08:30), inside the validity of both feeds; the panel
# data are from May-June 2024 but the timetables are current ones, as in 04.2 (the feeds are not archived back to 2024).
DEPARTURE = "2026-11-10 08:30"
ACCESS_EGRESS_WALK_MIN = 30
GTFS_DIR = DATA / "gtfs" / f"london-{AREA}"; GTFS_DIR.mkdir(parents=True, exist_ok=True)
BODS_FEED_ID = "f-bus~dft~gov~uk"            # Bus Open Data Service aggregate in transitio's index: buses, Underground, DLR, trams
GB_TRANSIT_RELEASE = "feed-2026-10-25"       # gb-transit (Planar Network): National Rail + TfL tube, DLR, trams, river, cable car
GB_TRANSIT_URL = f"https://github.com/planarnetwork/gb-transit/releases/download/{GB_TRANSIT_RELEASE}/gtfs-rail-and-tfl.zip"
GB_TRANSIT_GTFS = DATA / "gtfs" / "london" / f"gb-transit-rail-and-tfl-{GB_TRANSIT_RELEASE}.zip"   # already on disk from 04.2
MODES = STREET_MODES + ["transit"]

RUN_FLOWS = os.environ.get("LOCO_RUN_FLOWS", "0") == "1"       # step 1 (full node); the routing job leaves it off and reads OD_FILE
RUN_ROUTING = os.environ.get("LOCO_RUN_ROUTING", "1") == "1"   # steps 3-8

## 1. Flows: home and work hexagon per user, summed to pairs

The same DuckDB query as the city notebooks and the representativeness notebook. Home per month = the cell with most night signals; work
per month = the other cell with most working-hour signals on working days (bank holidays excluded); one record per user, the latest month
in which both qualify. Commuters are users with `home_days >= 10` and `work_days >= 4` whose home **and** work cell lie in the area.
The output is one row per (home cell, work cell) with the number of users, which is all the routing needs; no user-level table is kept.
This step needs the full node (the two monthly tables are 38 and 44 GB) and runs only when `OD_FILE` is missing and `RUN_FLOWS` is on.

In [ ]:
fua_gdf = gpd.read_file(FUA_GEOJSON).set_index("URAU_CODE")
if AREA == "fua":
    AREA_POLYGON = fua_gdf.loc[FUA_CODE].geometry
else:
    city_gdf = gpd.read_file(CITY_GEOJSON)
    AREA_POLYGON = city_gdf.loc[city_gdf.URAU_CODE.str.match(r"UK1(0[1-9]|[12]\d|3[0-3])C1")].union_all()
area_cells = sorted(h3.geo_to_cells(AREA_POLYGON, H3_RES))
print(f"{AREA}: {gpd.GeoSeries([AREA_POLYGON], crs=4326).to_crs(METRIC_CRS).area.iloc[0] / 1e6:,.0f} km², {len(area_cells):,} level-{H3_RES} cells")

if OD_FILE.exists():
    od = pd.read_parquet(OD_FILE); print(f"loaded {OD_FILE.name}")
elif RUN_FLOWS:
    NIGHT = " + ".join(f"h{h}" for h in NIGHT_HOURS); WORKH = " + ".join(f"h{h}" for h in WORK_HOURS)
    holiday_sql = " OR ".join(f"(month = '{m}' AND day IN ({', '.join(map(str, d))}))" for m, d in HOLIDAYS.items() if d) or "false"
    wd = f"dow IN ({', '.join(map(str, WORKDAYS))}) AND NOT ({holiday_sql})"
    tmp_dir = Path(os.environ["TMPDIR"]) / "duck_tmp_routing"
    mem_gb = int(os.environ.get("SLURM_MEM_PER_NODE", "16000")) // 1000
    con = duckdb.connect()
    con.execute(f"SET threads = {int(os.environ.get('SLURM_CPUS_PER_TASK', os.cpu_count()))}; SET memory_limit = '{int(mem_gb * 0.8)}GB'; "
                f"SET temp_directory = '{tmp_dir}'; SET preserve_insertion_order = false; SET enable_progress_bar = false;")
    con.execute("CREATE VIEW ud AS " + " UNION ALL ".join(f"SELECT '{m}' AS month, * FROM read_parquet('{p}')" for m, p in UD_PATHS.items()))
    con.execute(f"""CREATE TABLE home AS SELECT userid, month, h3_8 AS home_cell, home_days FROM (
        SELECT userid, month, h3_8, count(*) AS home_days,
               row_number() OVER (PARTITION BY userid, month ORDER BY sum(n_night) DESC, sum(n_total) DESC, h3_8) AS rn
        FROM (SELECT userid, month, h3_8, n_total, ({NIGHT}) AS n_night FROM ud) WHERE n_night > 0 GROUP BY userid, month, h3_8) WHERE rn = 1""")
    con.execute(f"""CREATE TABLE work AS SELECT userid, month, h3_8 AS work_cell, work_days FROM (
        SELECT u.userid, u.month, u.h3_8, count(*) AS work_days,
               row_number() OVER (PARTITION BY u.userid, u.month ORDER BY sum(u.n_work) DESC, sum(u.n_total) DESC, u.h3_8) AS rn
        FROM (SELECT userid, month, day, h3_8, n_total, ({WORKH}) AS n_work FROM ud WHERE {wd}) u JOIN home h USING (userid, month)
        WHERE u.h3_8 <> h.home_cell AND u.n_work > 0 GROUP BY u.userid, u.month, u.h3_8) WHERE rn = 1""")
    con.register("area_cells", pd.DataFrame({"cell": area_cells}))
    od = con.execute(f"""
        SELECT home_cell, work_cell, count(*) AS users FROM (
            SELECT h.userid, h.home_cell, w.work_cell,
                   row_number() OVER (PARTITION BY h.userid ORDER BY (h.home_days >= {H_MIN} AND w.work_days >= {W_MIN}) DESC, h.month DESC) AS rn
            FROM home h JOIN work w USING (userid, month)
            WHERE h.home_days >= {H_MIN} AND w.work_days >= {W_MIN}
        ) WHERE rn = 1 AND home_cell IN (SELECT cell FROM area_cells) AND work_cell IN (SELECT cell FROM area_cells)
        GROUP BY home_cell, work_cell""").df()
    con.close()
    od.to_parquet(OD_FILE, index=False); print(f"saved {OD_FILE}")
else:
    raise SystemExit("OD_FILE is missing: run routing/jobs/london-flows.sbatch first (LOCO_RUN_FLOWS=1 on a full node)")

od = od.rename(columns={"home_cell": "from_id", "work_cell": "to_id"})
print(f"{len(od):,} hexagon pairs carrying {od.users.sum():,} commuters; {od.from_id.nunique():,} home hexagons, {od.to_id.nunique():,} work hexagons")
print(f"pairs with one user: {(od.users == 1).mean():.1%} of pairs, {od.loc[od.users == 1, 'users'].sum() / od.users.sum():.1%} of commuters")
od.users.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_frame().T

In [ ]:
# The strongest pairs, with the straight-line distance between the hexagon centres.
def centre_xy(cells):
    lat, lon = zip(*[h3.cell_to_latlng(c) for c in cells])
    return gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs=4326).to_crs(METRIC_CRS)

strongest = od.nlargest(STRONGEST, "users").reset_index(drop=True)
a, b = centre_xy(strongest.from_id), centre_xy(strongest.to_id)
strongest["straight_km"] = np.hypot(a.x.values - b.x.values, a.y.values - b.y.values) / 1000
strongest.round(1)

## 2. Routing points

A router starts and ends at a point, so each hexagon gets one. For a **home** hexagon the point is the population-weighted mean of the
census Output Area centroids inside it (the OA table of the representativeness notebook), so it sits where the residents live rather than
in a park or a reservoir; a hexagon with no OA centroid keeps its centre. For a **work** hexagon the centre is used: there is no
job-location layer at this resolution, and a level-8 hexagon is small (0.65 km² here, about 900 m across) so the centre is never far from
anything inside it. A hexagon that is both home and work for someone gets two points, one of each kind, and the pair "same hexagon"
is routed between them rather than set to zero.

In [ ]:
from shapely.geometry import Polygon

oa = pd.read_parquet(OA_CENTROIDS)
oa["cell"] = [h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(oa.lat.values, oa.lon.values)]
oa_xy = gpd.GeoSeries(gpd.points_from_xy(oa.lon, oa.lat), crs=4326).to_crs(METRIC_CRS)
oa["wx"], oa["wy"] = oa_xy.x.values * oa.population, oa_xy.y.values * oa.population
weighted = oa.groupby("cell")[["wx", "wy", "population"]].sum()
weighted = weighted[weighted.population > 0]
weighted["x"], weighted["y"] = weighted.wx / weighted.population, weighted.wy / weighted.population

home_users = od.groupby("from_id").users.sum()
home_cells = home_users[home_users >= MIN_USERS_HOME].index.tolist()
work_cells = sorted(od.to_id.unique())

def hexagon(cell):
    # the hexagon as a polygon in the metric CRS (h3 v4 gives (lat, lng) vertices)
    return Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(cell)])

def point_table(cells, kind):
    centres = centre_xy(cells)
    t = pd.DataFrame({"cell": cells, "kind": kind, "cx": centres.x.values, "cy": centres.y.values})
    if kind == "home":
        t = t.join(weighted[["x", "y", "population"]], on="cell")
        t["method"] = np.where(t.x.notna(), "OA population-weighted mean", "hexagon centre (no OA)")
        t["x"] = t.x.fillna(t.cx); t["y"] = t.y.fillna(t.cy)
    else:
        t["x"], t["y"], t["method"] = t.cx, t.cy, "hexagon centre"
    return t

points = pd.concat([point_table(home_cells, "home"), point_table(work_cells, "work")], ignore_index=True)
points["id"] = points.kind.str[0] + ":" + points.cell          # "h:88194a..." and "w:88194a...": one hexagon can carry both kinds
points = gpd.GeoDataFrame(points, geometry=gpd.GeoSeries([hexagon(c) for c in points.cell], crs=4326).to_crs(METRIC_CRS))
points["point"] = gpd.points_from_xy(points.x, points.y, crs=METRIC_CRS)
points["shift_m"] = np.hypot(points.x - points.cx, points.y - points.cy)
print(f"{len(home_cells):,} home hexagons (>= {MIN_USERS_HOME} commuters, {home_users[home_cells].sum() / home_users.sum():.1%} of commuters), {len(work_cells):,} work hexagons")
display(points.groupby(["kind", "method"]).size().to_frame("hexagons"))
print("shift of the home point from the hexagon centre (m):")
points.loc[points.kind == "home", "shift_m"].describe(percentiles=[0.5, 0.9]).round(0).to_frame().T

## 3. Street network

The streets come from OpenStreetMap. The Greater London extract of 04.2 stops at the boroughs, so the England extract (Geofabrik, about
1.3 GB, downloaded by the job if missing) is read instead, clipped by cafein to the bounding box of the area. cafein builds the walking,
cycling and driving graphs once and saves them on scratch; later runs load the file.

In [ ]:
if RUN_ROUTING:
    if not OSM_PBF.exists():
        OSM_PBF.parent.mkdir(parents=True, exist_ok=True); urllib.request.urlretrieve(OSM_URL, OSM_PBF)
    print(OSM_PBF.name, f"{OSM_PBF.stat().st_size / 1e9:.2f} GB")
    bounding_box = tuple(gpd.GeoSeries([AREA_POLYGON], crs=4326).buffer(0.02).total_bounds)   # about 2 km of margin around the area
    started = time.time()
    if STREET_ARTIFACT.exists():
        streets = StreetNetwork.load(STREET_ARTIFACT); print("loaded", STREET_ARTIFACT)
    else:
        streets = StreetNetwork.from_osm(OSM_PBF, modes=("walk", "bicycle", "car"), country=COUNTRY, bounding_box=bounding_box)
        streets.save(STREET_ARTIFACT); print("built and saved", STREET_ARTIFACT)
    print(f"{time.time() - started:.0f} s; bounding box {np.round(bounding_box, 3)}; vertices {streets.vertex_count:,}, edges {streets.edge_count:,}")

## 4. Snapping

A route begins and ends on a street, so each point is moved to the closest piece of street **inside its own hexagon**, at most 1 km
away; otherwise the hexagon is left without a point and listed. Every segment of the network counts, whether or not cars may use it.
Same rule as London 04.2 and the US metros.

In [ ]:
if RUN_ROUTING:
    started = time.time()
    edges = streets.streets_gdf[["geometry"]].to_crs(METRIC_CRS)
    edges_in_hex = gpd.sjoin(edges, points[["id", "geometry"]], how="inner", predicate="intersects")
    print(f"{len(edges):,} edges, {len(edges_in_hex):,} edge-hexagon matches, {time.time() - started:.0f} s")
    snapped = []
    for pid, polygon, pt in zip(points.id, points.geometry, points.point):
        inside = edges_in_hex.loc[edges_in_hex.id == pid, "geometry"]
        if len(inside) == 0:
            snapped.append({"id": pid, "snap_distance_m": np.nan, "snapped": None}); continue
        streets_inside = inside.intersection(polygon).union_all()
        on_street = nearest_points(pt, streets_inside)[1]
        snapped.append({"id": pid, "snap_distance_m": pt.distance(on_street), "snapped": on_street})
    points = points.drop(columns=[c for c in ["snap_distance_m", "snapped", "within_rule"] if c in points.columns]).merge(pd.DataFrame(snapped), on="id")
    points["within_rule"] = points.snap_distance_m <= SNAP_DISTANCE_M
    routing_points = gpd.GeoDataFrame({"id": points.loc[points.within_rule, "id"].to_numpy()},
                                      geometry=points.loc[points.within_rule, "snapped"].to_numpy(), crs=METRIC_CRS).to_crs(4326)
    routing_points.to_parquet(RESULTS / "routing_points.parquet", index=False)
    print(f"{len(routing_points):,} routing points; {points.snap_distance_m.isna().sum()} hexagons with no street inside, "
          f"{(~points.within_rule).sum()} left out under the {SNAP_DISTANCE_M} m rule, {time.time() - started:.0f} s")
    print("distance moved (m):"); display(points.snap_distance_m.describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_frame().T)
    if (~points.within_rule).any():
        display(points.loc[~points.within_rule, ["id", "kind", "snap_distance_m"]])

In [ ]:
if RUN_ROUTING:
    fig, ax = plt.subplots(figsize=(10, 9))
    gpd.GeoSeries([AREA_POLYGON], crs=4326).to_crs(METRIC_CRS).plot(ax=ax, facecolor="none", edgecolor="#999999", linewidth=0.6)
    points.loc[points.kind == "home"].plot(column="snap_distance_m", ax=ax, cmap="YlOrRd", vmin=0, vmax=SNAP_DISTANCE_M, edgecolor="none",
                                           legend=True, legend_kwds={"shrink": 0.6, "label": "home point to snapped point (m)"}, missing_kwds={"color": "#444444"})
    ax.set_title(f"{CITY} ({AREA}): snapping distance of the home points, per hexagon", loc="left", fontsize=10); ax.set_axis_off(); fig.tight_layout()

## 5. Street modes

cafein computes the travel time from every home point to every work point four times: car as if the roads were empty, car with the
junction delays of the morning rush, bicycle, e-bike. The tables are streamed to scratch in shards and reused by later runs (`resume`).
Then the hexagon pairs of step 1 are joined to them: every pair gets its users, the straight-line distance between its points, and
the four times in minutes. cafein stops a street route at 120 minutes, so a longer bicycle trip counts as not reached.

Size: with about 5,000 home points and 5,000 work points this is 25 million pairs per mode, ten times the US metros. The first run
should use the smoke test below (`SMOKE = True`, the strongest pairs only) to check the inputs and time one origin before the full matrix.

In [ ]:
SMOKE = os.environ.get("LOCO_SMOKE", "0") == "1"

def matrix_complete(path):
    manifest = Path(path) / "manifest.json"
    if not manifest.is_file():
        return False
    shards = json.loads(manifest.read_text())["shards"]
    return bool(shards) and all(shard["completed"] for shard in shards)

if RUN_ROUTING:
    origins = routing_points[routing_points.id.str.startswith("h:")].reset_index(drop=True)
    destinations = routing_points[routing_points.id.str.startswith("w:")].reset_index(drop=True)
    if SMOKE:
        keep = strongest.assign(from_id="h:" + strongest.from_id, to_id="w:" + strongest.to_id)
        origins = origins[origins.id.isin(keep.from_id)]; destinations = destinations[destinations.id.isin(keep.to_id)]
    print(f"{len(origins):,} origins x {len(destinations):,} destinations" + (" (smoke test)" if SMOKE else ""))
    for mode in STREET_MODES:
        output = MATRICES / (STREET_QUERIES[mode]["folder"] + ("-smoke" if SMOKE else ""))
        if matrix_complete(output):
            print(f"{mode:12} complete -> {output}"); continue
        started = time.time()
        TravelTimeMatrix.to_parquet(streets, origins, destinations, output=output, output_time_units="seconds",
                                    resume=output.exists(), **STREET_QUERIES[mode]["query"])
        print(f"{mode:12} {time.time() - started:6.0f} s -> {output}")

In [ ]:
if RUN_ROUTING:
    od_times = od.assign(from_id="h:" + od.from_id, to_id="w:" + od.to_id)
    od_times = od_times[od_times.from_id.isin(routing_points.id) & od_times.to_id.isin(routing_points.id)].copy()
    xy = routing_points.set_index("id").to_crs(METRIC_CRS).geometry
    od_times["straight_km"] = np.hypot(xy.x.loc[od_times.from_id].values - xy.x.loc[od_times.to_id].values,
                                       xy.y.loc[od_times.from_id].values - xy.y.loc[od_times.to_id].values) / 1000
    for mode in STREET_MODES:
        m = read_shards(MATRICES / (STREET_QUERIES[mode]["folder"] + ("-smoke" if SMOKE else "")))[["from_id", "to_id", "travel_time"]]
        m = m.rename(columns={"travel_time": f"{mode}_min"}); m[f"{mode}_min"] = m[f"{mode}_min"] / 60
        od_times = od_times.merge(m, on=["from_id", "to_id"], how="left")
    if SMOKE:
        od_times = od_times.merge(strongest.assign(from_id="h:" + strongest.from_id, to_id="w:" + strongest.to_id)[["from_id", "to_id"]])
    od_times.to_parquet(RESULTS / ("od_times_smoke.parquet" if SMOKE else "od_times.parquet"), index=False)
    print(f"{len(od_times):,} pairs between routing points, {od_times.users.sum():,} commuters")
    display(od_times.sort_values("users", ascending=False).head(STRONGEST).round(1))

## 6. Timetables

Two GTFS sources, as in 04.2, because London has no single feed:

- **Bus Open Data Service** (DfT): the national aggregate, fetched through transitio, which crops it to the area polygon, validates it and
  repairs what it can. It carries the buses of every operator and TfL's Underground, DLR and trams.
- **gb-transit** (Planar Network): National Rail, Overground, Elizabeth line and TfL's own timetables (tube, DLR, trams, river, cable car)
  converted from the official CIF and TransXChange files; a pinned nightly release, national coverage, no cropping needed.

The Underground appears in both; cafein keeps both copies, and the duplicates cost only build time since the same trip never beats itself.
The table lists what each file contains and whether it runs on the departure day.

In [ ]:
ROUTE_TYPE_NAMES = {0: "tram", 1: "metro", 2: "rail", 3: "bus", 4: "ferry", 5: "cable tram", 6: "aerial", 7: "funicular", 11: "trolleybus", 12: "monorail"}

def gtfs_summary(path):
    with zipfile.ZipFile(path) as archive:
        names = set(archive.namelist())
        agency = pd.read_csv(archive.open("agency.txt")); routes = pd.read_csv(archive.open("routes.txt"))
        stops = pd.read_csv(archive.open("stops.txt"), usecols=["stop_id"]); trips = pd.read_csv(archive.open("trips.txt"), usecols=["trip_id"])
        dates = []
        if "calendar.txt" in names:
            cal = pd.read_csv(archive.open("calendar.txt"), dtype=str); dates += [cal.start_date.min(), cal.end_date.max()]
        if "calendar_dates.txt" in names:
            exc = pd.read_csv(archive.open("calendar_dates.txt"), dtype=str); dates += [exc.date.min(), exc.date.max()]
    by_type = routes.route_type.map(lambda t: ROUTE_TYPE_NAMES.get(int(t), f"type {t}")).value_counts()
    return {"file": Path(path).name, "agencies": len(agency), "routes": len(routes), "stops": len(stops), "trips": len(trips),
            "first_date": min(dates) if dates else None, "last_date": max(dates) if dates else None, **{f"routes_{k}": v for k, v in by_type.items()}}

if RUN_ROUTING:
    started = time.time()
    fetched = transitio.fetch(aoi=AREA_POLYGON, feeds=[BODS_FEED_ID], when=pd.Timestamp(DEPARTURE).date(), osm=False, directory=GTFS_DIR)
    print(f"transitio fetch done in {time.time() - started:.0f} s; skipped: {fetched.skipped}")
    for report in fetched.reports:
        print("report:", str(report)[:400])
    if not GB_TRANSIT_GTFS.exists():
        urllib.request.urlretrieve(GB_TRANSIT_URL, GB_TRANSIT_GTFS)
    GTFS_FILES = [Path(str(f)) for f in fetched.feeds] + [GB_TRANSIT_GTFS]
    display(pd.DataFrame([gtfs_summary(p) for p in GTFS_FILES]).set_index("file").T)

## 7. Transit

cafein links the stops of both feeds to the streets, so a trip can walk to a stop, ride, change and walk again. Built once from the
feeds and the extract (clipped to the area) and saved on scratch. The transit time from every home point to every work point is then
computed for a departure at 08:30 on the Tuesday, with the walk to the first stop and from the last stop capped at 30 minutes as in the
US runs, and joined next to the street modes.

In [ ]:
if RUN_ROUTING:
    started = time.time()
    if TRANSIT_ARTIFACT.exists():
        transit_network = TransportNetwork.load(TRANSIT_ARTIFACT); print("loaded", TRANSIT_ARTIFACT)
    else:
        transit_network = TransportNetwork.from_gtfs(GTFS_FILES, osm_pbf=OSM_PBF, street_modes=("walk", "bicycle", "car"),
                                                     country=COUNTRY, bounding_box=bounding_box)
        transit_network.save(TRANSIT_ARTIFACT); print("built and saved", TRANSIT_ARTIFACT)
    print(f"{time.time() - started:.0f} s; {transit_network.stop_count:,} stops, {transit_network.trip_count:,} trips")

    output = MATRICES / (f"transit-walk{ACCESS_EGRESS_WALK_MIN}" + ("-smoke" if SMOKE else ""))
    if matrix_complete(output):
        print(f"transit      complete -> {output}")
    else:
        started = time.time()
        TravelTimeMatrix.to_parquet(transit_network, origins, destinations, output=output, departure=DEPARTURE,
                                    max_walking_time=ACCESS_EGRESS_WALK_MIN, output_time_units="seconds", resume=output.exists())
        print(f"transit      {time.time() - started:6.0f} s -> {output}")
    m = read_shards(output)[["from_id", "to_id", "travel_time"]].rename(columns={"travel_time": "transit_min"}); m["transit_min"] = m.transit_min / 60
    od_times = od_times.drop(columns=[c for c in ["transit_min"] if c in od_times.columns]).merge(m, on=["from_id", "to_id"], how="left")
    od_times.to_parquet(RESULTS / ("od_times_smoke.parquet" if SMOKE else "od_times.parquet"), index=False)

## 8. Summary

Commuter-weighted share reached and minutes by mode over the hexagon pairs, the median per home hexagon, and the maps. The same
summary functions as the US metros, with `users` in place of `workers`.

In [ ]:
def weighted_percentiles(values, weights, percentiles):
    order = np.argsort(values); cumulative = np.cumsum(weights[order]) / weights.sum()
    return [values[order][np.searchsorted(cumulative, p / 100)] for p in percentiles]

def mode_summary(table, modes):
    out = {}
    for mode in modes:
        reached = table[f"{mode}_min"].notna()
        minutes, w, straight = table.loc[reached, f"{mode}_min"].values, table.loc[reached, "users"].values.astype(float), table.loc[reached, "straight_km"].values
        out[mode] = {"share of commuters reached": w.sum() / table.users.sum(),
                     "mean minutes, commuter-weighted": np.average(minutes, weights=w),
                     "median minutes, commuter-weighted": weighted_percentiles(minutes, w, [50])[0],
                     "p90 minutes, commuter-weighted": weighted_percentiles(minutes, w, [90])[0],
                     "straight-line km per hour": straight[minutes > 0].dot(w[minutes > 0]) / (minutes[minutes > 0] / 60).dot(w[minutes > 0])}
    return pd.DataFrame(out)

if RUN_ROUTING:
    print("commutes between the routed hexagons, by mode:"); display(mode_summary(od_times, MODES).round(2))

In [ ]:
if RUN_ROUTING and not SMOKE:
    per_hex = pd.DataFrame({"residents": od_times.groupby("from_id").users.sum()})
    for mode in MODES:
        col = f"{mode}_min"; reached = od_times[od_times[col].notna()]
        per_hex[f"{mode}_median_min"] = reached.groupby("from_id").apply(lambda r: weighted_percentiles(r[col].values, r.users.values.astype(float), [50])[0], include_groups=False)
    per_hex = per_hex.rename_axis("id").reset_index(); per_hex["h3_8"] = per_hex.id.str[2:]
    per_hex.to_csv(RESULTS / "median_minutes_by_home_hexagon.csv", index=False)
    print("median minutes per home hexagon, across hexagons:"); display(per_hex[[f"{m}_median_min" for m in MODES]].describe(percentiles=[0.25, 0.5, 0.75]).round(1).T)

    mapped = points.loc[points.kind == "home", ["id", "geometry"]].merge(per_hex, on="id", how="left")
    fig, axes = plt.subplots(ncols=2, figsize=(16, 7))
    for ax, mode in zip(axes, ["car", "transit"]):
        col = f"{mode}_median_min"
        mapped.plot(column=col, ax=ax, cmap="viridis", vmin=0, vmax=np.nanpercentile(mapped[col], 98), edgecolor="none",
                    legend=True, legend_kwds={"shrink": 0.7, "label": "minutes"}, missing_kwds={"color": "#ececea"})
        gpd.GeoSeries([AREA_POLYGON], crs=4326).to_crs(METRIC_CRS).plot(ax=ax, facecolor="none", edgecolor="#999999", linewidth=0.6)
        ax.set_title(f"{CITY} ({AREA}): median commute by {mode}, per home hexagon", loc="left", fontsize=10); ax.set_axis_off()
    fig.tight_layout()

## Notes

- Jobs (submit from `roihu-cpu.csc.fi`, from `~/locomizer_signals`): `sbatch routing/jobs/london-flows.sbatch` once (full node, step 1,
  about 10 min), then `sbatch routing/jobs/london-routing.sbatch` (8 cores, 96 GB; set `LOCO_SMOKE=1` in the script for the first run).
- The notebook reads the Locomizer tables from project_2022768 and writes the networks, matrices and GTFS files next to the other
  cities under project_2010423, so the cafein project can reuse them; the result tables go to `results/GBR/london_<period>/routing/`.
- Open points: the timetables are from November 2026 while the panel is from May–June 2024; the e-bike and bicycle see no slopes
  (no elevation model); `MIN_USERS_HOME` can thin the origin set if the full matrix is too slow.